# Prompt Engineering Lab

This notebook completes the TechFlow Solutions prompt engineering lab. It builds baseline zero-shot prompts, tests them repeatedly, documents failure patterns, improves the prompts with structure and constraints, then applies few-shot and Chain-of-Thought techniques for final evaluation.

## Step 1: Setup Your Environment

Run the setup cell first. If `openai` is missing, install it in the active notebook environment with `%pip install openai pandas` and rerun the cell. Keep your API key in the `OPENAI_API_KEY` environment variable; do not paste secrets into the notebook.

In [1]:
%pip install openai pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import re
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
from openai import OpenAI

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

from dotenv import load_dotenv
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY is not set. Set it before running this lab.")

client = OpenAI()

def call_openai(prompt, *, temperature=0.7, max_tokens=500, json_mode=False):
    """Call OpenAI with one prompt and return plain response text."""
    kwargs = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "You are a precise customer-service AI assistant."},
            {"role": "user", "content": prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content.strip()

def run_prompt(label, prompt, runs, *, temperature=0.7, max_tokens=500, json_mode=False, workers=5):
    """Run the same prompt multiple times and return a DataFrame of responses."""
    records = []
    started = time.time()
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {
            pool.submit(
                call_openai,
                prompt,
                temperature=temperature,
                max_tokens=max_tokens,
                json_mode=json_mode,
            ): i
            for i in range(1, runs + 1)
        }
        for future in as_completed(futures):
            run_number = futures[future]
            try:
                response = future.result()
                error = ""
            except Exception as exc:
                response = ""
                error = repr(exc)
            records.append(
                {
                    "label": label,
                    "run": run_number,
                    "response": response,
                    "error": error,
                }
            )
    df = pd.DataFrame(records).sort_values("run").reset_index(drop=True)
    output_path = RESULTS_DIR / f"{label}_{runs}_runs.csv"
    df.to_csv(output_path, index=False)
    print(f"Completed {runs} runs for {label} in {time.time() - started:.1f}s -> {output_path}")
    return df

def normalize_text(text):
    return re.sub(r"\s+", " ", str(text).strip().lower())

def sentiment_label(text):
    normalized = normalize_text(text)
    for label in ["positive", "negative", "neutral"]:
        if normalized == label or normalized.startswith(label):
            return label
    return "invalid"

def is_valid_json(text):
    try:
        json.loads(text)
        return True
    except Exception:
        return False

def product_format_signature(text):
    lines = [line.strip() for line in str(text).splitlines() if line.strip()]
    words = len(str(text).split())
    has_title = any(line.lower().startswith("title:") for line in lines)
    has_bullets = any(line.startswith("-") for line in lines)
    has_cta = "call to action:" in str(text).lower()
    word_bucket = "short" if words < 60 else "target" if words <= 120 else "long"
    return f"title={has_title}|bullets={has_bullets}|cta={has_cta}|words={word_bucket}"

def summarize_consistency(label, df, kind):
    successful = df[df["error"].eq("")].copy()
    if successful.empty:
        return {
            "label": label,
            "task_type": kind,
            "runs": len(df),
            "success_rate": 0,
            "consistency_percent": 0,
            "unique_outputs": 0,
            "failure_notes": "All runs failed; inspect error column.",
        }

    if kind == "sentiment":
        normalized = successful["response"].map(sentiment_label)
        notes = "Invalid label or extra text" if "invalid" in set(normalized) else "Labels stayed in allowed set"
    elif kind == "extraction":
        normalized = successful["response"].map(lambda x: "valid_json" if is_valid_json(x) else "invalid_json")
        notes = "Some output was not valid JSON" if "invalid_json" in set(normalized) else "All output parsed as JSON"
    else:
        normalized = successful["response"].map(product_format_signature)
        notes = "Format signatures compare structure, not exact marketing copy"

    counts = Counter(normalized)
    most_common_count = counts.most_common(1)[0][1]
    return {
        "label": label,
        "task_type": kind,
        "runs": len(df),
        "success_rate": round(len(successful) / len(df) * 100, 1),
        "consistency_percent": round(most_common_count / len(successful) * 100, 1),
        "unique_outputs": successful["response"].map(normalize_text).nunique(),
        "failure_notes": notes,
    }

print("Setup complete. OpenAI client and helper functions are ready.")

Setup complete. OpenAI client and helper functions are ready.


## Step 2: Create Initial Prompts

These v1 prompts are intentionally simple zero-shot prompts. They establish the baseline before structure and examples are added.

In [2]:
sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""

product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""

extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

print("Sentiment Analysis Result:")
print(call_openai(sentiment_prompt_v1))
print("\nProduct Description Result:")
print(call_openai(product_prompt_v1))
print("\nData Extraction Result:")
print(call_openai(extraction_prompt_v1))

Sentiment Analysis Result:
This customer message can be classified as positive feedback or a customer compliment.

Product Description Result:
**Product Description: Wireless Precision Mouse**

Elevate your productivity with our Wireless Precision Mouse, designed for seamless performance and ultimate comfort. Priced at just $29.99, this sleek and stylish mouse is perfect for both work and play.

**Key Features:**

- **Wireless Freedom:** Enjoy a clutter-free workspace with advanced 2.4GHz wireless technology for a reliable connection up to 33 feet away.
  
- **Ergonomic Design:** The contoured shape fits naturally in your hand, reducing strain and allowing for extended use without discomfort.
  
- **Customizable Buttons:** Personalize your experience with programmable buttons that let you execute commands with a single click, enhancing your workflow.
  
- **Long-lasting Battery Life:** Equipped with an energy-efficient design, this mouse boasts an impressive battery life that lasts for

### Initial observations

- Sentiment analysis:
it returned extra explanation instead of simple word Positive.

- Product description:
it has invented unsupported features such as battery life and programmable buttons.

- Data extraction:
correct information extracted but in human redable format not in Json format.

#Part 2: Diagnosing Failures - Systematic Testing

## Steps 3-5: Diagnose Failures with 5, 10, and 15 Runs

Run each v1 prompt multiple times and inspect whether responses stay consistent in format and content.

## Testing 5 Runs

In [3]:
v1_results = {}
for runs in [5]:
    v1_results[("sentiment_v1", runs)] = run_prompt("sentiment_v1", sentiment_prompt_v1, runs, temperature=0.7, max_tokens=60)
    v1_results[("product_v1", runs)] = run_prompt("product_v1", product_prompt_v1, runs, temperature=0.7, max_tokens=220)
    v1_results[("extraction_v1", runs)] = run_prompt("extraction_v1", extraction_prompt_v1, runs, temperature=0.7, max_tokens=220)

v1_analysis = pd.DataFrame([
    summarize_consistency("sentiment_v1", v1_results[("sentiment_v1", 5)], "sentiment"),
    summarize_consistency("product_v1", v1_results[("product_v1", 5)], "product"),
    summarize_consistency("extraction_v1", v1_results[("extraction_v1", 5)], "extraction"),
])
v1_analysis.to_csv(RESULTS_DIR / "v1_failure_analysis.csv", index=False)
v1_analysis

Completed 5 runs for sentiment_v1 in 1.3s -> results\sentiment_v1_5_runs.csv
Completed 5 runs for product_v1 in 3.5s -> results\product_v1_5_runs.csv
Completed 5 runs for extraction_v1 in 1.2s -> results\extraction_v1_5_runs.csv


,label,task_type,runs,success_rate,consistency_percent,unique_outputs,failure_notes
0,sentiment_v1,sentiment,5,100.0,80.0,5,Invalid label or extra text
1,product_v1,product,5,100.0,100.0,5,"Format signatures compare structure, not exact..."
2,extraction_v1,extraction,5,100.0,100.0,5,Some output was not valid JSON


### 5-run observations Notes
-success_rate =100%, which means this test has every API request completed.

- unique_outputs = 5, means it has produced every task with 5 different extract responses.

sentiment has 80% consistency, which means it has atleast one invalid or overly verbose response.
Product description has 100% consistency, which means, its broad structural signature matched. its wording still varied five times.
Data Extractions has also 100% consistency,which is misleading because, it was consistently showing invalid JSON. only human redable text.


## Testing 10-Runs

In [4]:
v1_results = {}
for runs in [10]:
    v1_results[("sentiment_v1", runs)] = run_prompt("sentiment_v1", sentiment_prompt_v1, runs, temperature=0.7, max_tokens=60)
    v1_results[("product_v1", runs)] = run_prompt("product_v1", product_prompt_v1, runs, temperature=0.7, max_tokens=220)
    v1_results[("extraction_v1", runs)] = run_prompt("extraction_v1", extraction_prompt_v1, runs, temperature=0.7, max_tokens=220)

v1_analysis = pd.DataFrame([
    summarize_consistency("sentiment_v1", v1_results[("sentiment_v1", 10)], "sentiment"),
    summarize_consistency("product_v1", v1_results[("product_v1", 10)], "product"),
    summarize_consistency("extraction_v1", v1_results[("extraction_v1", 10)], "extraction"),
])
v1_analysis.to_csv(RESULTS_DIR / "v1_10runs_failure_analysis.csv", index=False)
v1_analysis

Completed 10 runs for sentiment_v1 in 2.3s -> results\sentiment_v1_10_runs.csv
Completed 10 runs for product_v1 in 7.5s -> results\product_v1_10_runs.csv
Completed 10 runs for extraction_v1 in 1.9s -> results\extraction_v1_10_runs.csv


,label,task_type,runs,success_rate,consistency_percent,unique_outputs,failure_notes
0,sentiment_v1,sentiment,10,100.0,100.0,6,Invalid label or extra text
1,product_v1,product,10,100.0,100.0,10,"Format signatures compare structure, not exact..."
2,extraction_v1,extraction,10,100.0,100.0,9,Some output was not valid JSON


### 10-run observations
-Sentiment: 6 unique responses; outputs still included extra text or invalid labels.

-Product: 10 unique responses—every response differed.

-Extraction: 9 unique responses and some/all outputs were not valid JSON.
100% consistency does not necessarily mean correctness here. It can mean responses consistently fell into the same broad category—for example, consistently invalid JSON.

## Testing 15 runs:

In [5]:
v1_results = {}
for runs in [15]:
    v1_results[("sentiment_v1", runs)] = run_prompt("sentiment_v1", sentiment_prompt_v1, runs, temperature=0.7, max_tokens=60)
    v1_results[("product_v1", runs)] = run_prompt("product_v1", product_prompt_v1, runs, temperature=0.7, max_tokens=220)
    v1_results[("extraction_v1", runs)] = run_prompt("extraction_v1", extraction_prompt_v1, runs, temperature=0.7, max_tokens=220)

v1_analysis = pd.DataFrame([
    summarize_consistency("sentiment_v1", v1_results[("sentiment_v1", 15)], "sentiment"),
    summarize_consistency("product_v1", v1_results[("product_v1", 15)], "product"),
    summarize_consistency("extraction_v1", v1_results[("extraction_v1", 15)], "extraction"),
])
v1_analysis.to_csv(RESULTS_DIR / "v1_15runs_failure_analysis.csv", index=False)
v1_analysis

Completed 15 runs for sentiment_v1 in 3.1s -> results\sentiment_v1_15_runs.csv
Completed 15 runs for product_v1 in 13.7s -> results\product_v1_15_runs.csv
Completed 15 runs for extraction_v1 in 2.8s -> results\extraction_v1_15_runs.csv


,label,task_type,runs,success_rate,consistency_percent,unique_outputs,failure_notes
0,sentiment_v1,sentiment,15,100.0,93.3,12,Invalid label or extra text
1,product_v1,product,15,100.0,100.0,15,"Format signatures compare structure, not exact..."
2,extraction_v1,extraction,15,100.0,100.0,11,Some output was not valid JSON


## 15-run test analysis


Variation increased again:

###Sentiment:
 12 unique responses; 93.3% shared the dominant broad category, but formatting still failed. The sentiment prompt is producing synonyms instead of the required label positive, which makes automated classification unreliable.
-All 15 responses are semantically positive.
-Only run 15 begins with the expected label.
-The other 14 add explanations or alternative labels.Therefore, semantic accuracy is high, but output-format reliability is poor.

#Product:
 15 unique responses—every generation differed. The product responses show three important failure patterns:
- Invented facts: 2.4 GHz connectivity, 33-foot range, adjustable DPI, battery type, included receiver, and specific battery life were never provided.
- Inconsistent branding and structure: names such as “SwiftClick,” “Wireless Precision Mouse,” and “Wireless Comfort Mouse” vary between runs.
- Truncated output: several responses end mid-sentence may be because the generation has reached the max_tokens=220 limit.

#Extraction:
 11 unique responses and invalid JSON remained a problem. The extraction is semantically correct, but structurally unreliable:
- No response is valid JSON.
- Field names vary: Item Number, Order Number, Item Ordered, and others.
- Formatting varies between plain bullets, bold Markdown, and introductory text.
- Runs 8 and 15 remove the # from the item number.
- Values vary slightly, such as Damaged versus Damaged packaging.


### Failure Analysis Notes

After running the cell above, documenting the concrete patterns I saw here:

| Prompt | Observed failures | Why it matters |
|---|---|---|
| Sentiment v1 | May include explanations instead of one label; may use casing or punctuation inconsistently. | A production classifier needs a stable label that downstream code can parse. |
| Product v1 | Marketing copy length, tone, and format can vary widely. | Product pages need predictable structure and brand style. |
| Extraction v1 | Output may be prose, bullets, or partial structured data instead of machine-readable JSON. | Data pipelines need consistent fields and valid syntax. |

## part 3: Iteration 1 - Rewriting Simple Prompts

## Steps 6-8: Iteration 1 - Adding Clarity, Constraints, and Format Requirements

In [3]:
sentiment_prompt_v2 = """
Classify the customer message as exactly one of these labels: positive, negative, neutral.

Rules:
- Respond with one lowercase word only.
- Do not explain your answer.
- Use neutral when the message is mixed, unclear, or only factual.

Customer message: "I love this product! It's exactly what I needed."
"""

product_prompt_v2 = """
Write a product description for the product below.

Product: Wireless mouse
Price: $29.99
Audience: Everyday laptop and office users

Requirements:
- 70-90 words
- Friendly, practical, and benefit-focused tone
- Mention wireless convenience, comfortable use, and value
- Avoid unsupported technical claims
- Return one paragraph only
"""

extraction_prompt_v2 = """
Extract structured information from the customer feedback below.

Return valid JSON only with these fields:
- item_number: string or null
- order_date: string or null
- delivery_sentiment: positive, negative, neutral, or null
- packaging_issue: true or false
- summary: one short sentence

Customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

print("Sentiment v2:", call_openai(sentiment_prompt_v2, temperature=0.2, max_tokens=20))
print("\nProduct v2:\n", call_openai(product_prompt_v2, temperature=0.5, max_tokens=180))
print("\nExtraction v2:\n", call_openai(extraction_prompt_v2, temperature=0.2, max_tokens=180, json_mode=True))

Sentiment v2: positive

Product v2:
 Experience the freedom of movement with our Wireless Mouse, designed for everyday laptop and office users. Priced at just $29.99, this mouse offers wireless convenience, eliminating the hassle of tangled cords while ensuring a clean workspace. Its ergonomic design provides comfortable use for hours, making it perfect for long work sessions or casual browsing. Enjoy the perfect blend of value and functionality as you navigate your tasks with ease and efficiency. Elevate your productivity today with this essential accessory!

Extraction v2:
 {
  "item_number": "12345",
  "order_date": "March 15th",
  "delivery_sentiment": "positive",
  "packaging_issue": true,
  "summary": "The delivery was fast but the packaging was damaged."
}


In [4]:
v2_results = {
    "sentiment_v2": run_prompt("sentiment_v2", sentiment_prompt_v2, 15, temperature=0.2, max_tokens=20),
    "product_v2": run_prompt("product_v2", product_prompt_v2, 15, temperature=0.5, max_tokens=180),
    "extraction_v2": run_prompt("extraction_v2", extraction_prompt_v2, 15, temperature=0.2, max_tokens=180, json_mode=True),
}

v2_analysis = pd.DataFrame([
    summarize_consistency("sentiment_v2", v2_results["sentiment_v2"], "sentiment"),
    summarize_consistency("product_v2", v2_results["product_v2"], "product"),
    summarize_consistency("extraction_v2", v2_results["extraction_v2"], "extraction"),
])
v2_analysis.to_csv(RESULTS_DIR / "v2_failure_analysis.csv", index=False)
v2_analysis

Completed 15 runs for sentiment_v2 in 2.6s -> results\sentiment_v2_15_runs.csv
Completed 15 runs for product_v2 in 6.6s -> results\product_v2_15_runs.csv
Completed 15 runs for extraction_v2 in 5.9s -> results\extraction_v2_15_runs.csv


,label,task_type,runs,success_rate,consistency_percent,unique_outputs,failure_notes
0,sentiment_v2,sentiment,15,100.0,100.0,1,Labels stayed in allowed set
1,product_v2,product,15,100.0,100.0,15,"Format signatures compare structure, not exact..."
2,extraction_v2,extraction,15,100.0,100.0,1,All output parsed as JSON


### Version 2 observations



#Sentiment:
93.3% vs V2 100%	Exactly one valid label in every run.

#Product:
	both version has consistently 100% structure output.	Key improvement:Constrained structure and no unsupported claims requested.

#Extraction:
	in V1 100% invalid-JSON category,	wheras in V2 100% valid JSON	significant improvement in V2: All 15 outputs parsed successfully


## Part 4:

## Steps 9-11: Iteration 2 - Few-Shot and Chain-of-Thought Techniques

The v3 prompts add examples and stricter structure. For extraction, the prompt asks the model to reason internally and return only final JSON so the reasoning helps quality without breaking machine parsing.

# V3 Test

In [6]:
sentiment_prompt_v3 = """
Classify the customer message as exactly one lowercase label: positive, negative, or neutral.
Return the label only.

Examples:
Message: "The agent fixed my issue quickly and was very kind."
Answer: positive

Message: "My order arrived late and nobody replied to my email."
Answer: negative

Message: "I received the invoice yesterday."
Answer: neutral

Message: "I love this product! It's exactly what I needed."
Answer:
"""

product_prompt_v3 = """
Create a concise product description using the exact format shown in the examples.

Rules:
- Keep the total output between 75 and 105 words.
- Use a friendly, practical, benefit-focused tone.
- Do not invent unsupported specifications.
- Return exactly these sections: Title, Description, Key benefits, Call to action.
- Use three bullet points under Key benefits.

Example 1:
Product: Insulated travel mug
Price: $18.99
Output:
Title: Insulated Travel Mug - $18.99
Description: Keep your drink close and your routine moving with an insulated mug built for commutes, desk days, and quick errands.
Key benefits:
- Helps drinks stay enjoyable longer
- Easy to carry during busy mornings
- Practical value for everyday use
Call to action: Add it to your daily setup today.

Example 2:
Product: Desk lamp
Price: $24.99
Output:
Title: Desk Lamp - $24.99
Description: Brighten your workspace with a simple desk lamp that supports reading, studying, and focused work without taking over your desk.
Key benefits:
- Clean look for home or office spaces
- Useful for work, reading, and study time
- Affordable upgrade for everyday productivity
Call to action: Make your workspace easier to use.

Product: Wireless mouse
Price: $29.99
Output:
"""

extraction_prompt_v3 = """
Extract information from customer feedback. Think through the feedback internally, then return only valid JSON.

JSON schema:
{
  "item_number": "string or null",
  "order_date": "string or null",
  "delivery_sentiment": "positive, negative, neutral, or null",
  "packaging_issue": "boolean",
  "issue_type": "string or null",
  "summary": "one short sentence"
}

Extraction rules:
- Use the customer's wording for dates when no year is given.
- delivery_sentiment refers only to delivery, not the whole message.
- packaging_issue is true only when packaging damage or packaging quality is mentioned.
- If a field is missing, use null.
- Return JSON only; no markdown, no explanation.

Example:
Feedback: "Order #A778 arrived two days late. The box was crushed, but the product works."
Output:
{
  "item_number": "A778",
  "order_date": null,
  "delivery_sentiment": "negative",
  "packaging_issue": true,
  "issue_type": "crushed box",
  "summary": "The order arrived late and the box was crushed."
}

Feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
Output:
"""

In [7]:
v3_results = {
    "sentiment_v3": run_prompt("sentiment_v3", sentiment_prompt_v3, 15, temperature=0, max_tokens=20),
    "product_v3": run_prompt("product_v3", product_prompt_v3, 15, temperature=0.2, max_tokens=220),
    "extraction_v3": run_prompt("extraction_v3", extraction_prompt_v3, 15, temperature=0, max_tokens=220, json_mode=True),
}

v3_analysis = pd.DataFrame([
    summarize_consistency("sentiment_v3", v3_results["sentiment_v3"], "sentiment"),
    summarize_consistency("product_v3", v3_results["product_v3"], "product"),
    summarize_consistency("extraction_v3", v3_results["extraction_v3"], "extraction"),
])
v3_analysis.to_csv(RESULTS_DIR / "v3_failure_analysis.csv", index=False)
v3_analysis

Completed 15 runs for sentiment_v3 in 2.4s -> results\sentiment_v3_15_runs.csv
Completed 15 runs for product_v3 in 5.6s -> results\product_v3_15_runs.csv
Completed 15 runs for extraction_v3 in 6.2s -> results\extraction_v3_15_runs.csv


,label,task_type,runs,success_rate,consistency_percent,unique_outputs,failure_notes
0,sentiment_v3,sentiment,15,100.0,100.0,1,Labels stayed in allowed set
1,product_v3,product,15,100.0,86.7,12,"Format signatures compare structure, not exact..."
2,extraction_v3,extraction,15,100.0,100.0,1,All output parsed as JSON


### Version 3 observations

Sentiment: 100% consistency; one exact output across 15 runs.

Product: 86.7% structural consistency; 12 unique descriptions. This is above the lab’s 80% target and shows few-shot examples made the format more stable.

Extraction: 100% valid JSON and one exact output across 15 runs.

overall, few-shot examples has improved sentiment and product-format consistency, while the reasoning instructions plus schema/example produced consistently valid extraction JSON.

# Part 5: Tuning for Different Tasks

### Step 12: Task Variations

Test whether the strongest prompts transfer to related customer-service cases.

In [8]:
sentiment_variations = [
    "The replacement arrived today and works perfectly.",
    "I waited 40 minutes for support and still have the same problem.",
    "Can you confirm whether my warranty expires next month?",
]

product_variations = [
    {"product": "USB-C laptop stand", "price": "$39.99"},
    {"product": "Noise-reducing office headset", "price": "$59.99"},
]

extraction_variations = [
    "Order #ZX-991 arrived on April 2. Delivery was delayed and the outer box was torn.",
    "I bought item 777 last week. Shipping was quick and everything was packed well.",
]

variation_records = []

for message in sentiment_variations:
    prompt = sentiment_prompt_v3.replace('"I love this product! It\'s exactly what I needed."', f'"{message}"')
    variation_records.append({"task": "sentiment", "input": message, "output": call_openai(prompt, temperature=0, max_tokens=20)})

for product in product_variations:
    prompt = product_prompt_v3.replace("Product: Wireless mouse\nPrice: $29.99", f"Product: {product['product']}\nPrice: {product['price']}")
    variation_records.append({"task": "product", "input": json.dumps(product), "output": call_openai(prompt, temperature=0.2, max_tokens=220)})

for feedback in extraction_variations:
    prompt = extraction_prompt_v3.replace('Feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."', f'Feedback: "{feedback}"')
    variation_records.append({"task": "extraction", "input": feedback, "output": call_openai(prompt, temperature=0, max_tokens=220, json_mode=True)})

variation_df = pd.DataFrame(variation_records)
variation_df.to_csv(RESULTS_DIR / "task_variations.csv", index=False)
variation_df

,task,input,output
0,sentiment,The replacement arrived today and works perfec...,positive
1,sentiment,I waited 40 minutes for support and still have...,negative
2,sentiment,Can you confirm whether my warranty expires ne...,neutral
3,product,"{""product"": ""USB-C laptop stand"", ""price"": ""$3...",Title: USB-C Laptop Stand - $39.99 \nDescript...
4,product,"{""product"": ""Noise-reducing office headset"", ""...",Title: Noise-Reducing Office Headset - $59.99 ...
5,extraction,Order #ZX-991 arrived on April 2. Delivery was...,"{\n ""item_number"": ""ZX-991"",\n ""order_date"":..."
6,extraction,I bought item 777 last week. Shipping was quic...,"{\n ""item_number"": ""777"",\n ""order_date"": ""l..."


### Task variation results

The task variations has worked correctly:
- Sentiment handled clearly positive, negative, and factual-question inputs.

- Product descriptions adapted to two new products while preserving the required format.

- Extraction handled a hyphenated order number and a relative date such as “last week.”

relative dates like “last week” are preserved as text, not converted into an exact calendar date. That is appropriate without a known reference date.

## Step 13: Final Evaluation and Comparison

Combining the v1, v2, and v3 analysis tables into one comparison report. The goal is for v3 prompts to show more than 80% consistency across 15 runs.

In [14]:
def load_results(filename):
    return pd.read_csv(RESULTS_DIR / filename).fillna("")

v1_analysis = pd.DataFrame([
    summarize_consistency(
        "sentiment_v1",
        load_results("sentiment_v1_15_runs.csv"),
        "sentiment",
    ),
    summarize_consistency(
        "product_v1",
        load_results("product_v1_15_runs.csv"),
        "product",
    ),
    summarize_consistency(
        "extraction_v1",
        load_results("extraction_v1_15_runs.csv"),
        "extraction",
    ),
])

v2_analysis = pd.DataFrame([
    summarize_consistency(
        "sentiment_v2",
        load_results("sentiment_v2_15_runs.csv"),
        "sentiment",
    ),
    summarize_consistency(
        "product_v2",
        load_results("product_v2_15_runs.csv"),
        "product",
    ),
    summarize_consistency(
        "extraction_v2",
        load_results("extraction_v2_15_runs.csv"),
        "extraction",
    ),
])

v3_analysis = pd.DataFrame([
    summarize_consistency(
        "sentiment_v3",
        load_results("sentiment_v3_15_runs.csv"),
        "sentiment",
    ),
    summarize_consistency(
        "product_v3",
        load_results("product_v3_15_runs.csv"),
        "product",
    ),
    summarize_consistency(
        "extraction_v3",
        load_results("extraction_v3_15_runs.csv"),
        "extraction",
    ),
])

print("Rebuilt analysis tables from saved 15-run CSV files.")

Rebuilt analysis tables from saved 15-run CSV files.


C:\Users\deuts\AppData\Local\Temp\ipykernel_16000\2089338759.py:2: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  return pd.read_csv(RESULTS_DIR / filename).fillna("")
C:\Users\deuts\AppData\Local\Temp\ipykernel_16000\2089338759.py:2: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  return pd.read_csv(RESULTS_DIR / filename).fillna("")
C:\Users\deuts\AppData\Local\Temp\ipykernel_16000\2089338759.py:2: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  return pd.read_csv(RESULTS_DIR / filename).fillna("")
C:\Users\deuts\AppData\Local\Temp\

In [15]:
comparison = pd.concat([v1_analysis, v2_analysis, v3_analysis], ignore_index=True)
comparison["version"] = comparison["label"].str.extract(r"_(v\d)$")
comparison["prompt"] = comparison["label"].str.replace(r"_v\d$", "", regex=True)
comparison = comparison[
    ["prompt", "version", "runs", "success_rate", "consistency_percent", "unique_outputs", "failure_notes"]
].sort_values(["prompt", "version"])
comparison.to_csv(RESULTS_DIR / "final_prompt_comparison.csv", index=False)
comparison

,prompt,version,runs,success_rate,consistency_percent,unique_outputs,failure_notes
2,extraction,v1,15,100.0,100.0,11,Some output was not valid JSON
5,extraction,v2,15,100.0,100.0,1,All output parsed as JSON
8,extraction,v3,15,100.0,100.0,1,All output parsed as JSON
1,product,v1,15,100.0,100.0,15,"Format signatures compare structure, not exact..."
4,product,v2,15,100.0,100.0,15,"Format signatures compare structure, not exact..."
7,product,v3,15,100.0,86.7,12,"Format signatures compare structure, not exact..."
0,sentiment,v1,15,100.0,93.3,12,Invalid label or extra text
3,sentiment,v2,15,100.0,100.0,1,Labels stayed in allowed set
6,sentiment,v3,15,100.0,100.0,1,Labels stayed in allowed set


## final comparison

Key conclusion: Version 3 is the final best set overall:

- Sentiment improved from 93.3% (invalid extra text) to 100% valid labels in V2 and V3.

- Extraction improved from consistently non-JSON V1 outputs to 100% valid JSON in V2 and V3.

- Product V3 achieved 86.7% format consistency, above the 80% target. V1/V2’s 100% only reflects the notebook’s broad format-signature metric, not truthful product facts or a controlled required template. V3 is more production-ready because it specifies the exact sections, examples, and claim restrictions.

In [16]:
final_prompts = {
    "sentiment_prompt_v3": sentiment_prompt_v3.strip(),
    "product_prompt_v3": product_prompt_v3.strip(),
    "extraction_prompt_v3": extraction_prompt_v3.strip(),
}

with open(RESULTS_DIR / "final_prompts.json", "w", encoding="utf-8") as file:
    json.dump(final_prompts, file, indent=2)

proof_notes = """
# Lab Run Notes

After running the notebook, copy the final comparison table and one representative query/output into lab_proof.md.
The generated CSV files in the results folder provide evidence for the 5, 10, and 15-run tests.
""".strip()

with open(RESULTS_DIR / "run_notes.md", "w", encoding="utf-8") as file:
    file.write(proof_notes + "\n")

print("Saved final prompts and run notes in the results folder.")

Saved final prompts and run notes in the results folder.


## Techniques That Worked Best

- Sentiment analysis improved most from few-shot examples, explicit labels, and `temperature=0`.
- Product description generation improved from fixed sections, length limits, style rules, and examples that demonstrate the expected format.
- Data extraction improved from JSON mode, a defined schema, field-level rules, and internal Chain-of-Thought instructions that still return only parseable JSON.